# Airline Route Profitability, SQL Analysis

This notebook builds a SQLite database from the raw flight data and runs a full
SQL analysis on it: route profitability, seasonality, fleet performance, demand,
cost structure, and two reusable views. Every query result below is generated by
running that query live against the database in this notebook.

Run the cells in order from top to bottom. The database is rebuilt from the raw
CSV each time this notebook runs, so results always reflect the current data.


## Setup

In [1]:

import sqlite3
import csv
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

BASE_DIR = Path(r"/home/claude/sql_nb_test/GitHub Repository/Airline Route Profitability")

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Project folder not found: {BASE_DIR}\n"
        f"Edit BASE_DIR above to point at your actual project folder."
    )

csv_candidates = [
    p for p in BASE_DIR.rglob("*.csv")
    if any(k in p.name.lower() for k in ("airline", "route", "profitab"))
]
if not csv_candidates:
    raise FileNotFoundError(f"No matching CSV found under {BASE_DIR}")
DATA_PATH = csv_candidates[0]
DB_PATH = BASE_DIR / "airline.db"
OUT_DIR = BASE_DIR / "outputs" / "sql_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data file:", DATA_PATH)
print("Database will be created at:", DB_PATH)


Data file: /home/claude/sql_nb_test/GitHub Repository/Airline Route Profitability/Airline_Route_Profitability.csv
Database will be created at: /home/claude/sql_nb_test/GitHub Repository/Airline Route Profitability/airline.db


## Building the database

The table schema is defined explicitly rather than inferred, so the column
types match what the data represents: text for identifiers and categories,
integers for counts, real numbers for money, ratios and hours, and a date
column for the flight date.

In [2]:

CREATE_TABLE_SQL = '''
CREATE TABLE flights (
    Flight_Number            TEXT,
    Flight_Date               DATE,
    Origin                     TEXT,
    Destination                TEXT,
    Route                      TEXT,
    Aircraft_Type              TEXT,
    Aircraft_Capacity          INTEGER,
    Passengers                 INTEGER,
    Load_Factor                REAL,
    Flight_Hours               REAL,
    Season                     TEXT,
    Route_Category             TEXT,
    Demand_Level               TEXT,
    Ticket_Revenue             REAL,
    Ancillary_Revenue          REAL,
    Total_Revenue              REAL,
    Fuel_Cost                  REAL,
    Maintenance_Cost           REAL,
    Crew_Cost                  REAL,
    Depreciation_Cost          REAL,
    Insurance_Cost             REAL,
    Airport_Fees               REAL,
    Catering_Cost              REAL,
    Handling_Cost              REAL,
    Navigation_Fees            REAL,
    Sales_Distribution_Cost    REAL,
    Passenger_Service_Cost     REAL,
    Overhead_Cost              REAL,
    Marketing_Cost             REAL,
    IT_Systems_Cost            REAL,
    Total_Cost                 REAL,
    Profit                     REAL,
    Profit_Margin              REAL
);
'''

INSERT_SQL = f"INSERT INTO flights VALUES ({', '.join(['?'] * 33)})"

INTEGER_COLS = {"Aircraft_Capacity", "Passengers"}
REAL_COLS = {
    "Load_Factor", "Flight_Hours", "Ticket_Revenue", "Ancillary_Revenue",
    "Total_Revenue", "Fuel_Cost", "Maintenance_Cost", "Crew_Cost",
    "Depreciation_Cost", "Insurance_Cost", "Airport_Fees", "Catering_Cost",
    "Handling_Cost", "Navigation_Fees", "Sales_Distribution_Cost",
    "Passenger_Service_Cost", "Overhead_Cost", "Marketing_Cost",
    "IT_Systems_Cost", "Total_Cost", "Profit", "Profit_Margin",
}

def cast_value(column, value):
    if value == "" or value is None:
        return None
    if column in INTEGER_COLS:
        return int(value)
    if column in REAL_COLS:
        return float(value)
    return value

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute(CREATE_TABLE_SQL)

with open(DATA_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = [tuple(cast_value(col, row[col]) for col in columns) for row in reader]

cur.executemany(INSERT_SQL, rows)
conn.commit()

cur.execute("CREATE INDEX idx_route ON flights(Route)")
cur.execute("CREATE INDEX idx_date ON flights(Flight_Date)")
cur.execute("CREATE INDEX idx_aircraft ON flights(Aircraft_Type)")
conn.commit()

row_count = cur.execute("SELECT COUNT(*) FROM flights").fetchone()[0]
print(f"Rows loaded into 'flights' table: {row_count}")


Rows loaded into 'flights' table: 7974


In [3]:

def run_query(sql, save_as=None):
    '''Run a query, return it as a DataFrame, and optionally save it to
    outputs/sql_results.'''
    df = pd.read_sql_query(sql, conn)
    if save_as:
        df.to_csv(OUT_DIR / f"{save_as}.csv", index=False)
    return df


## Section 1, Data overview

Row count, date range, and how many distinct routes and aircraft types are in the data.

In [4]:

run_query('''
    SELECT
        COUNT(*)                      AS total_flights,
        MIN(Flight_Date)              AS first_date,
        MAX(Flight_Date)              AS last_date,
        COUNT(DISTINCT Route)         AS distinct_routes,
        COUNT(DISTINCT Aircraft_Type) AS distinct_aircraft_types
    FROM flights;
''', save_as="01_overview")


,total_flights,first_date,last_date,distinct_routes,distinct_aircraft_types
0,7974,2024-01-01,2024-12-31,30,6


Flights and revenue by season.

In [5]:

run_query('''
    SELECT
        Season,
        COUNT(*)                     AS flights,
        ROUND(SUM(Total_Revenue), 2) AS total_revenue,
        ROUND(AVG(Load_Factor), 3)   AS avg_load_factor
    FROM flights
    GROUP BY Season
    ORDER BY total_revenue DESC;
''', save_as="02_flights_by_season")


,Season,flights,total_revenue,avg_load_factor
0,Shoulder,2646,8.273671e+08,0.822
1,Peak,2002,6.918343e+08,0.870
2,Low,2001,4.775488e+08,0.719
3,Normal,1325,3.750061e+08,0.782


## Section 2, Route profitability

Revenue, cost, profit, and margin for every route.

In [6]:

run_query('''
    SELECT
        Route,
        Route_Category,
        COUNT(*)                     AS flights,
        ROUND(AVG(Load_Factor), 3)   AS avg_load_factor,
        ROUND(SUM(Total_Revenue), 2) AS total_revenue,
        ROUND(SUM(Total_Cost), 2)    AS total_cost,
        ROUND(SUM(Profit), 2)        AS total_profit,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin
    FROM flights
    GROUP BY Route, Route_Category
    ORDER BY total_profit DESC;
''', save_as="03_route_summary")


,Route,Route_Category,flights,avg_load_factor,total_revenue,total_cost,total_profit,avg_profit_margin
0,DXB-FRA,Long Haul,325,0.850,2.048922e+08,1.054007e+08,99491502.54,46.08
1,DXB-SIN,Long Haul,320,0.850,2.039344e+08,1.129396e+08,90994879.30,42.07
2,DXB-CDG,Long Haul,323,0.853,2.039859e+08,1.138073e+08,90178585.87,41.27
3,DXB-HKG,Long Haul,332,0.851,2.182465e+08,1.287146e+08,89531878.76,38.67
4,DXB-JFK,Long Haul,328,0.851,2.137032e+08,1.724657e+08,41237491.55,15.70
5,DXB-BKK,Long Haul,212,0.740,9.495948e+07,5.439553e+07,40563948.78,39.91
6,DXB-KUL,Long Haul,216,0.728,9.641778e+07,5.993626e+07,36481521.99,34.49
7,DXB-SYD,Long Haul,327,0.849,2.053551e+08,1.742288e+08,31126294.37,9.98
8,DXB-BOM,Medium Haul,325,0.848,6.903962e+07,4.496927e+07,24070355.00,31.16
9,DXB-DEL,Medium Haul,332,0.852,6.803000e+07,4.697777e+07,21052229.00,27.44


Routes that lose money overall, worst first.

In [7]:

run_query('''
    SELECT
        Route,
        COUNT(*)                     AS flights,
        ROUND(SUM(Profit), 2)        AS total_profit,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin
    FROM flights
    GROUP BY Route
    HAVING SUM(Profit) < 0
    ORDER BY total_profit ASC;
''', save_as="04_loss_making_routes")


,Route,flights,total_profit,avg_profit_margin
0,DXB-CAI,201,-8190219.81,-83.96
1,DXB-AMM,219,-7912976.35,-76.29
2,DXB-LAX,206,-7905366.81,-15.59
3,DXB-LHR,344,-6699578.75,-13.44
4,DXB-SFO,210,-5994202.11,-12.71
5,DXB-JED,205,-3876933.27,-68.19
6,DXB-RUH,216,-2411299.99,-41.20
7,DXB-KWI,341,-630031.24,-10.63


Rank routes by total profit within each route category, using a window function partitioned by category.

In [8]:

run_query('''
    WITH route_totals AS (
        SELECT
            Route,
            Route_Category,
            SUM(Profit) AS total_profit
        FROM flights
        GROUP BY Route, Route_Category
    )
    SELECT
        Route,
        Route_Category,
        ROUND(total_profit, 2) AS total_profit,
        RANK() OVER (
            PARTITION BY Route_Category
            ORDER BY total_profit DESC
        ) AS rank_in_category
    FROM route_totals
    ORDER BY Route_Category, rank_in_category;
''', save_as="05_rank_by_category")


,Route,Route_Category,total_profit,rank_in_category
0,DXB-FRA,Long Haul,99491502.54,1
1,DXB-SIN,Long Haul,90994879.30,2
2,DXB-CDG,Long Haul,90178585.87,3
3,DXB-HKG,Long Haul,89531878.76,4
4,DXB-JFK,Long Haul,41237491.55,5
5,DXB-BKK,Long Haul,40563948.78,6
6,DXB-KUL,Long Haul,36481521.99,7
7,DXB-SYD,Long Haul,31126294.37,8
8,DXB-MEL,Long Haul,7747732.31,9
9,DXB-ORD,Long Haul,3868119.92,10


Each route's profit compared against the portfolio average route profit, using a subquery.

In [9]:

run_query('''
    SELECT
        Route,
        ROUND(SUM(Profit), 2) AS route_profit,
        ROUND(
            SUM(Profit) - (SELECT SUM(Profit) FROM flights) / COUNT(DISTINCT Route)
        , 2) AS diff_from_avg_route_profit
    FROM flights
    GROUP BY Route
    ORDER BY route_profit DESC;
''', save_as="06_diff_from_avg")


,Route,route_profit,diff_from_avg_route_profit
0,DXB-FRA,99491502.54,-4.759892e+08
1,DXB-SIN,90994879.30,-4.844858e+08
2,DXB-CDG,90178585.87,-4.853021e+08
3,DXB-HKG,89531878.76,-4.859488e+08
4,DXB-JFK,41237491.55,-5.342432e+08
5,DXB-BKK,40563948.78,-5.349167e+08
6,DXB-KUL,36481521.99,-5.389991e+08
7,DXB-SYD,31126294.37,-5.443544e+08
8,DXB-BOM,24070355.00,-5.514103e+08
9,DXB-DEL,21052229.00,-5.544284e+08


## Section 3, Seasonality and time trends

Monthly revenue, cost, and profit.

In [10]:

run_query('''
    SELECT
        strftime('%Y-%m', Flight_Date) AS year_month,
        ROUND(SUM(Total_Revenue), 2)   AS revenue,
        ROUND(SUM(Total_Cost), 2)      AS cost,
        ROUND(SUM(Profit), 2)          AS profit
    FROM flights
    GROUP BY year_month
    ORDER BY year_month;
''', save_as="07_monthly_trend")


,year_month,revenue,cost,profit
0,2024-01,2.239072e+08,1.530918e+08,70815387.01
1,2024-02,2.307545e+08,1.572122e+08,73542224.03
2,2024-03,2.077939e+08,1.524257e+08,55368218.69
3,2024-04,2.069013e+08,1.497263e+08,57175063.76
4,2024-05,1.904638e+08,1.479124e+08,42551397.40
5,2024-06,1.613424e+08,1.424404e+08,18901917.20
6,2024-07,1.568650e+08,1.417203e+08,15144716.20
7,2024-08,1.593415e+08,1.415554e+08,17786040.02
8,2024-09,1.845423e+08,1.429820e+08,41560258.23
9,2024-10,1.990053e+08,1.478503e+08,51155075.53


Month over month profit change, using LAG to compare each month against the one before it.

In [11]:

run_query('''
    WITH monthly AS (
        SELECT
            strftime('%Y-%m', Flight_Date) AS year_month,
            SUM(Profit) AS profit
        FROM flights
        GROUP BY year_month
    )
    SELECT
        year_month,
        ROUND(profit, 2) AS profit,
        ROUND(profit - LAG(profit) OVER (ORDER BY year_month), 2) AS profit_change_vs_prev_month,
        ROUND(
            100.0 * (profit - LAG(profit) OVER (ORDER BY year_month))
            / NULLIF(LAG(profit) OVER (ORDER BY year_month), 0)
        , 2) AS profit_pct_change
    FROM monthly
    ORDER BY year_month;
''', save_as="08_month_over_month")


,year_month,profit,profit_change_vs_prev_month,profit_pct_change
0,2024-01,70815387.01,NaN,NaN
1,2024-02,73542224.03,2726837.02,3.85
2,2024-03,55368218.69,-18174005.34,-24.71
3,2024-04,57175063.76,1806845.07,3.26
4,2024-05,42551397.40,-14623666.36,-25.58
5,2024-06,18901917.20,-23649480.20,-55.58
6,2024-07,15144716.20,-3757201.00,-19.88
7,2024-08,17786040.02,2641323.82,17.44
8,2024-09,41560258.23,23774218.21,133.67
9,2024-10,51155075.53,9594817.30,23.09


Three month rolling average profit, using a window frame.

In [12]:

run_query('''
    WITH monthly AS (
        SELECT
            strftime('%Y-%m', Flight_Date) AS year_month,
            SUM(Profit) AS profit
        FROM flights
        GROUP BY year_month
    )
    SELECT
        year_month,
        ROUND(profit, 2) AS profit,
        ROUND(
            AVG(profit) OVER (
                ORDER BY year_month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            )
        , 2) AS rolling_3month_avg_profit
    FROM monthly
    ORDER BY year_month;
''', save_as="09_rolling_avg")


,year_month,profit,rolling_3month_avg_profit
0,2024-01,70815387.01,70815387.01
1,2024-02,73542224.03,72178805.52
2,2024-03,55368218.69,66575276.58
3,2024-04,57175063.76,62028502.16
4,2024-05,42551397.40,51698226.62
5,2024-06,18901917.20,39542792.79
6,2024-07,15144716.20,25532676.93
7,2024-08,17786040.02,17277557.81
8,2024-09,41560258.23,24830338.15
9,2024-10,51155075.53,36833791.26


Profit margin by season, ranked.

In [13]:

run_query('''
    SELECT
        Season,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin,
        RANK() OVER (ORDER BY AVG(Profit_Margin) DESC) AS margin_rank
    FROM flights
    GROUP BY Season;
''', save_as="10_season_rank")


,Season,avg_profit_margin,margin_rank
0,Peak,17.55,1
1,Shoulder,10.14,2
2,Normal,3.33,3
3,Low,-8.50,4


## Section 4, Fleet and aircraft performance

Aircraft level summary, including RASK and CASK, revenue and cost per available seat hour.

In [14]:

run_query('''
    SELECT
        Aircraft_Type,
        COUNT(*)                     AS flights,
        ROUND(AVG(Load_Factor), 3)   AS avg_load_factor,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin,
        ROUND(SUM(Total_Revenue) / SUM(Aircraft_Capacity * Flight_Hours), 2) AS rask,
        ROUND(SUM(Total_Cost) / SUM(Aircraft_Capacity * Flight_Hours), 2)    AS cask
    FROM flights
    GROUP BY Aircraft_Type
    ORDER BY avg_profit_margin DESC;
''', save_as="11_fleet_summary")


,Aircraft_Type,flights,avg_load_factor,avg_profit_margin,rask,cask
0,Airbus A380,793,0.848,23.18,152.24,105.18
1,Boeing 777-300ER,1690,0.807,17.26,128.17,97.30
2,Boeing 787-9,1818,0.804,9.48,140.35,100.57
3,Airbus A350-900,1700,0.759,1.04,105.13,89.17
4,Airbus A320,956,0.811,-7.87,140.07,133.63
5,Boeing 737-800,1017,0.813,-9.50,134.38,131.44


Best performing aircraft type per route, using ROW_NUMBER to keep only the top row in each group.

In [15]:

run_query('''
    WITH aircraft_route AS (
        SELECT
            Route,
            Aircraft_Type,
            AVG(Profit_Margin) AS avg_margin,
            ROW_NUMBER() OVER (
                PARTITION BY Route
                ORDER BY AVG(Profit_Margin) DESC
            ) AS rn
        FROM flights
        GROUP BY Route, Aircraft_Type
    )
    SELECT
        Route,
        Aircraft_Type AS best_aircraft,
        ROUND(avg_margin, 2) AS avg_profit_margin
    FROM aircraft_route
    WHERE rn = 1
    ORDER BY avg_profit_margin DESC;
''', save_as="12_best_aircraft_per_route")


,Route,best_aircraft,avg_profit_margin
0,DXB-FRA,Boeing 787-9,46.72
1,DXB-SIN,Boeing 777-300ER,43.71
2,DXB-CDG,Boeing 787-9,42.84
3,DXB-BKK,Airbus A350-900,41.89
4,DXB-HKG,Boeing 787-9,40.96
5,DXB-KUL,Airbus A350-900,35.51
6,DXB-BOM,Airbus A350-900,32.77
7,DXB-KHI,Airbus A320,31.50
8,DXB-DEL,Airbus A350-900,29.55
9,DXB-LHE,Airbus A350-900,19.37


## Section 5, Demand and load factor

Load factor bands built with CASE, and the average profit margin in each band.

In [16]:

run_query('''
    SELECT
        CASE
            WHEN Load_Factor < 0.60 THEN 'Under 60%'
            WHEN Load_Factor < 0.70 THEN '60-70%'
            WHEN Load_Factor < 0.80 THEN '70-80%'
            WHEN Load_Factor < 0.90 THEN '80-90%'
            ELSE '90% and above'
        END AS load_factor_band,
        COUNT(*)                     AS flights,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin
    FROM flights
    GROUP BY load_factor_band
    ORDER BY MIN(Load_Factor);
''', save_as="13_load_factor_bands")


,load_factor_band,flights,avg_profit_margin
0,Under 60%,39,-46.16
1,60-70%,1040,-23.51
2,70-80%,2867,-0.95
3,80-90%,2757,16.18
4,90% and above,1271,26.54


Demand level versus profitability.

In [17]:

run_query('''
    SELECT
        Demand_Level,
        COUNT(*)                     AS flights,
        ROUND(AVG(Load_Factor), 3)   AS avg_load_factor,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin
    FROM flights
    GROUP BY Demand_Level
    ORDER BY avg_profit_margin DESC;
''', save_as="14_demand_level")


,Demand_Level,flights,avg_load_factor,avg_profit_margin
0,High,4630,0.850,17.69
1,Medium,3344,0.734,-9.73


Quartile split of routes by average profit margin, using NTILE.

In [18]:

run_query('''
    WITH route_margin AS (
        SELECT
            Route,
            AVG(Profit_Margin) AS avg_margin
        FROM flights
        GROUP BY Route
    )
    SELECT
        Route,
        ROUND(avg_margin, 2) AS avg_profit_margin,
        NTILE(4) OVER (ORDER BY avg_margin DESC) AS profit_quartile
    FROM route_margin
    ORDER BY avg_margin DESC;
''', save_as="15_profit_quartiles")


,Route,avg_profit_margin,profit_quartile
0,DXB-FRA,46.08,1
1,DXB-SIN,42.07,1
2,DXB-CDG,41.27,1
3,DXB-BKK,39.91,1
4,DXB-HKG,38.67,1
5,DXB-KUL,34.49,1
6,DXB-BOM,31.16,1
7,DXB-KHI,30.53,1
8,DXB-DEL,27.44,2
9,DXB-LHE,18.72,2


## Section 6, Cost structure

Average direct, service, and indirect cost per flight.

In [19]:

run_query('''
    SELECT
        ROUND(AVG(Fuel_Cost + Maintenance_Cost + Crew_Cost + Depreciation_Cost + Insurance_Cost), 2) AS avg_direct_op_cost,
        ROUND(AVG(Airport_Fees + Catering_Cost + Handling_Cost + Navigation_Fees), 2)                AS avg_service_cost,
        ROUND(AVG(Sales_Distribution_Cost + Passenger_Service_Cost + Overhead_Cost + Marketing_Cost + IT_Systems_Cost), 2) AS avg_indirect_cost
    FROM flights;
''', save_as="16_cost_structure")


,avg_direct_op_cost,avg_service_cost,avg_indirect_cost
0,103938.51,18844.49,102502.41


Routes where fuel cost makes up the largest share of total cost.

In [20]:

run_query('''
    SELECT
        Route,
        ROUND(AVG(Fuel_Cost), 2)                           AS avg_fuel_cost,
        ROUND(AVG(Total_Cost), 2)                          AS avg_total_cost,
        ROUND(100.0 * AVG(Fuel_Cost) / AVG(Total_Cost), 2) AS fuel_pct_of_cost
    FROM flights
    GROUP BY Route
    ORDER BY fuel_pct_of_cost DESC
    LIMIT 10;
''', save_as="17_fuel_share")


,Route,avg_fuel_cost,avg_total_cost,fuel_pct_of_cost
0,DXB-LAX,117886.16,462734.43,25.48
1,DXB-SFO,120201.10,476499.42,25.23
2,DXB-ORD,104404.86,428764.73,24.35
3,DXB-MEL,97276.32,409291.73,23.77
4,DXB-SYD,126198.65,532809.68,23.69
5,DXB-JFK,122191.01,525809.97,23.24
6,DXB-LHR,60043.86,264070.24,22.74
7,DXB-CAI,20346.94,98848.49,20.58
8,DXB-AMM,18433.26,92973.53,19.83
9,DXB-HKG,76316.05,387694.54,19.68


## Section 7, Reusable views

The two views below stay in the database after this notebook runs, so they can
be queried directly in any future session without redefining them.

A view summarizing every route, ready to query directly.

In [21]:

conn.execute("DROP VIEW IF EXISTS route_summary")
conn.execute('''
    CREATE VIEW route_summary AS
    SELECT
        Route,
        Route_Category,
        Demand_Level,
        COUNT(*)                     AS flights,
        ROUND(AVG(Load_Factor), 3)   AS avg_load_factor,
        ROUND(SUM(Total_Revenue), 2) AS total_revenue,
        ROUND(SUM(Total_Cost), 2)    AS total_cost,
        ROUND(SUM(Profit), 2)        AS total_profit,
        ROUND(AVG(Profit_Margin), 2) AS avg_profit_margin
    FROM flights
    GROUP BY Route, Route_Category, Demand_Level;
''')
conn.commit()

run_query("SELECT * FROM route_summary ORDER BY total_profit DESC LIMIT 5;", save_as="18_route_summary_view")


,Route,Route_Category,Demand_Level,flights,avg_load_factor,total_revenue,total_cost,total_profit,avg_profit_margin
0,DXB-FRA,Long Haul,High,325,0.850,2.048922e+08,1.054007e+08,99491502.54,46.08
1,DXB-SIN,Long Haul,High,320,0.850,2.039344e+08,1.129396e+08,90994879.30,42.07
2,DXB-CDG,Long Haul,High,323,0.853,2.039859e+08,1.138073e+08,90178585.87,41.27
3,DXB-HKG,Long Haul,High,332,0.851,2.182465e+08,1.287146e+08,89531878.76,38.67
4,DXB-JFK,Long Haul,High,328,0.851,2.137032e+08,1.724657e+08,41237491.55,15.70


A view classifying each route into a profitability tier, using nested subqueries against the view above.

In [22]:

conn.execute("DROP VIEW IF EXISTS route_segments")
conn.execute('''
    CREATE VIEW route_segments AS
    SELECT
        Route,
        avg_load_factor,
        avg_profit_margin,
        CASE
            WHEN avg_profit_margin >= (SELECT AVG(avg_profit_margin) FROM route_summary)
                 AND avg_load_factor >= (SELECT AVG(avg_load_factor) FROM route_summary)
                THEN 'Star'
            WHEN avg_profit_margin >= (SELECT AVG(avg_profit_margin) FROM route_summary)
                 AND avg_load_factor < (SELECT AVG(avg_load_factor) FROM route_summary)
                THEN 'Cash Cow'
            WHEN avg_profit_margin < (SELECT AVG(avg_profit_margin) FROM route_summary)
                 AND avg_load_factor >= (SELECT AVG(avg_load_factor) FROM route_summary)
                THEN 'Question Mark'
            ELSE 'Underperformer'
        END AS segment
    FROM route_summary;
''')
conn.commit()

run_query('''
    SELECT segment, COUNT(*) AS routes
    FROM route_segments
    GROUP BY segment
    ORDER BY routes DESC;
''', save_as="19_route_segments")


,segment,routes
0,Star,11
1,Underperformer,10
2,Cash Cow,6
3,Question Mark,3


## Closing the connection

Every query result above was also saved as a CSV in `outputs/sql_results`, and
the two views, `route_summary` and `route_segments`, remain in `airline.db` for
later use.

In [23]:

conn.close()
print("Connection closed. Database saved at:", DB_PATH)
print("Query results saved in:", OUT_DIR)


Connection closed. Database saved at: /home/claude/sql_nb_test/GitHub Repository/Airline Route Profitability/airline.db
Query results saved in: /home/claude/sql_nb_test/GitHub Repository/Airline Route Profitability/outputs/sql_results
